# Robotic Warehouse DES Simulation: Scenario Analysis

This notebook is the analyst-facing companion to the project. It is intentionally lightweight: the production logic lives in `src/warehouse_sim`, while this notebook loads the repeatable experiment outputs and explains how to interpret them.


## Analysis Questions

1. How does throughput change as robot fleet size increases?
2. At what demand level do cycle time and SLA attainment start to degrade?
3. Is the system more likely to be robot constrained or station constrained?
4. Which scenarios should be recommended for deeper production testing?


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from warehouse_sim.experiments import run_experiment_suite

REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR


## Regenerate the Standard Experiment Suite

The experiment runner executes a scenario matrix, writes CSV outputs, and refreshes SVG charts. Use fewer replications while developing and the default setting for final reporting.


In [ ]:
outputs = run_experiment_suite(REPORTS_DIR, replications=5)
scenario_summary = outputs['scenario_summary']
fleet_summary = outputs['fleet_summary']
demand_summary = outputs['demand_summary']
scenario_summary.head()


## Executive Scenario Summary

The main table is scenario-grain: each row represents the average result across deterministic replications.


In [ ]:
display_cols = [
    'scenario_id', 'robot_count', 'station_count', 'arrival_rate_per_minute',
    'throughput_per_hour', 'avg_cycle_time_minutes', 'p90_cycle_time_minutes',
    'robot_utilization', 'station_utilization', 'sla_attainment_rate',
    'bottleneck_classification'
]
scenario_summary[display_cols].sort_values('scenario_id').round(3)


## Fleet Size Sensitivity

This view checks whether adding robots keeps improving throughput or reaches diminishing returns.

![Throughput by fleet size](../reports/figures/throughput_by_fleet_size.svg)


In [ ]:
fleet_summary[['robot_count', 'throughput_per_hour', 'avg_cycle_time_minutes', 'sla_attainment_rate', 'bottleneck_classification']].round(3)


## Demand Stress Test

This view shows the nonlinear latency behavior that appears as demand approaches capacity.

![Cycle time by demand](../reports/figures/cycle_time_by_demand.svg)


In [ ]:
demand_summary[['arrival_rate_per_minute', 'throughput_per_hour', 'avg_cycle_time_minutes', 'p90_cycle_time_minutes', 'sla_attainment_rate']].round(3)


## Recommendation Pattern

A strong recommendation from this model should not be based on throughput alone. It should consider throughput, SLA attainment, P90 cycle time, robot utilization, station utilization, and queue growth together.

A realistic next step would be to calibrate the synthetic parameters with historical event logs, then test dispatching policies and layout-aware routing.
